# Pocket TTS Farsi v2 ONNX - Voice Cloning Test

This notebook follows the author's `nimaone/persian_tts` ONNX pipeline and lets you upload your own reference WAV and synthesize Persian speech with that voice.

It installs ONNX Runtime GPU and prefers CUDA when available, with CPU fallback. The model was designed primarily for CPU execution, so GPU execution may not accelerate every operation or improve total speed.

The model is released under **CC-BY-NC-4.0**, so this notebook is for testing/non-commercial use.

## 1. Install dependencies

Installs the ONNX Runtime GPU package. The notebook prefers CUDA when ONNX Runtime exposes `CUDAExecutionProvider`, and falls back to CPU if CUDA is unavailable or session initialization fails.

In [ ]:
%pip uninstall -y -q onnxruntime onnxruntime-gpu
%pip install -q -U "huggingface_hub[cli]" onnxruntime-gpu scipy soundfile sentencepiece

## 2. Clone the author's repository

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/persian_tts')

if not REPO_DIR.exists():
    !git clone -q https://github.com/nimaone/persian_tts.git /content/persian_tts

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)

## 3. Download the ONNX model package

The ONNX package is about 491 MB and contains the inference graphs, constants, and manifest. citeturn0search2

In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR = REPO_DIR / 'model' / 'onnx'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id='Nimaone/pocket-tts-farsi-v2-onnx',
    local_dir=str(MODEL_DIR),
)

print('ONNX model downloaded to:', MODEL_DIR)

## 4. Download the two extra Farsi-v2 runtime assets

The ONNX package contains the ONNX graphs, but the author's `tts_onnx.py` pipeline also expects the original Farsi-v2 tokenizer and Persian normalizer under `model/v2/`. These are downloaded from the original Farsi-v2 model repository.

In [ ]:
from huggingface_hub import hf_hub_download

V2_DIR = REPO_DIR / 'model' / 'v2'
V2_DIR.mkdir(parents=True, exist_ok=True)

for filename in ['tokenizer_ph.model', 'normalize_fa.py']:
    hf_hub_download(
        repo_id='mehdi-hf/pocket-tts-farsi-v2',
        filename=filename,
        local_dir=str(V2_DIR),
    )

print('Runtime assets:', V2_DIR)

## 5. Load the ONNX TTS engine

The notebook configures ONNX Runtime to prefer CUDA when available. Unsupported operators may still execute on CPU, and GPU acceleration is not guaranteed to be faster for this model.

In [ ]:
import sys
import time
from pathlib import Path

import onnxruntime as ort

available_providers = ort.get_available_providers()
use_cuda = "CUDAExecutionProvider" in available_providers

print("ONNX Runtime version:", ort.__version__)
print("Available providers:", available_providers)
print("Preferred execution:", "CUDA + CPU fallback" if use_cuda else "CPU")

# Patch session creation before importing the project so all ONNX sessions
# prefer CUDA when it is available, while retaining a safe CPU fallback.
_original_inference_session = ort.InferenceSession

def _gpu_aware_inference_session(*args, **kwargs):
    if use_cuda:
        kwargs["providers"] = ["CUDAExecutionProvider", "CPUExecutionProvider"]
        try:
            session = _original_inference_session(*args, **kwargs)
            print("ONNX session providers:", session.get_providers())
            return session
        except Exception as exc:
            print("CUDA session initialization failed; falling back to CPU.")
            print("CUDA error:", str(exc)[:500])
            kwargs["providers"] = ["CPUExecutionProvider"]
            session = _original_inference_session(*args, **kwargs)
            print("ONNX session providers:", session.get_providers())
            return session

    kwargs["providers"] = ["CPUExecutionProvider"]
    session = _original_inference_session(*args, **kwargs)
    print("ONNX session providers:", session.get_providers())
    return session

ort.InferenceSession = _gpu_aware_inference_session

sys.path.insert(0, str(REPO_DIR / 'scripts'))

from tts_onnx import OnnxTts

tts = OnnxTts(seed=42)
print('ONNX TTS engine loaded.')

## 6. Upload your reference voice

Use a **short, clean WAV recording**, ideally around 5 seconds, with one speaker and little background noise. The runtime trims/resamples the reference voice as needed.

If you upload a file named something else, the notebook automatically uses the first uploaded audio file.

In [ ]:
from google.colab import files
from IPython.display import Audio, display
from pathlib import Path

uploaded = files.upload()

audio_extensions = {'.wav', '.mp3', '.m4a', '.flac', '.ogg'}

uploaded_audio = [
    name for name in uploaded
    if Path(name).suffix.lower() in audio_extensions
]

if not uploaded_audio:
    raise RuntimeError('No audio file was uploaded.')

audio_name = uploaded_audio[0]
audio_data = uploaded[audio_name]

VOICE_WAV = Path('/content') / audio_name
VOICE_WAV.write_bytes(audio_data)

print('Reference voice:', VOICE_WAV)
print('Size:', VOICE_WAV.stat().st_size, 'bytes')

display(Audio(data=audio_data, autoplay=False))

## 7. Generate Persian speech with your voice

Edit `TEXT` and run this cell. `pace=1.0` is the normal setting. The author's runtime supports `split` and `pack` modes; `split` is a good first test for a paragraph.

In [ ]:
!pip install -q "numpy<2.3"

In [ ]:
import time
import soundfile as sf
from IPython.display import Audio, display

TEXT = 'سلام، حال شما چطور است؟ این یک تست برای بررسی کیفیت کلونینگ صدای فارسی با مدل پاکت تی‌تی‌اس است.'
PACE = 1.0
MODE = 'split'

OUTPUT_WAV = REPO_DIR / 'output' / 'my_voice_test.wav'
OUTPUT_WAV.parent.mkdir(parents=True, exist_ok=True)

start = time.perf_counter()

audio = tts.synthesize_text(
    TEXT,
    str(VOICE_WAV),
    pace=PACE,
    mode=MODE,
)

elapsed = time.perf_counter() - start

sf.write(str(OUTPUT_WAV), audio, 24000)

duration = len(audio) / 24000

print(f'Output: {OUTPUT_WAV}')
print(f'Audio duration: {duration:.2f}s')
print(f'Generation time: {elapsed:.2f}s')
print(f'Real-time factor: {elapsed / duration:.2f}x')

display(Audio(filename=str(OUTPUT_WAV)))

## 8. Try your own text repeatedly

The model is already loaded, so you can change only `TEXT` and rerun this cell.

In [ ]:
TEXT = 'یکی بود یکی نبود و این داستانا. بعدش هم اینو بگم که من اینجا نیستماااا'

start = time.perf_counter()

audio = tts.synthesize_text(
    TEXT,
    str(VOICE_WAV),
    pace=1.0,
    mode='split',
)

elapsed = time.perf_counter() - start

output_path = REPO_DIR / 'output' / 'custom_test.wav'
sf.write(str(output_path), audio, 24000)

duration = len(audio) / 24000

print(
    f'Generated {duration:.2f}s in '
    f'{elapsed:.2f}s ({elapsed / duration:.2f}x real time)'
)

display(Audio(filename=str(output_path)))

## 10. Download the generated WAV


In [ ]:
from google.colab import files
import shutil

output_dir = REPO_DIR / 'output'
zip_path = REPO_DIR / 'tts_outputs.zip'

shutil.make_archive(
    str(zip_path.with_suffix('')),
    'zip',
    root_dir=str(output_dir),
)

print(f'Created: {zip_path}')
print(f'Files included:')

for file in sorted(output_dir.iterdir()):
    if file.is_file():
        print(f'  - {file.name}')

files.download(str(zip_path))

### Notes

- The notebook installs `onnxruntime-gpu` and prefers `CUDAExecutionProvider` when it is available.
- `CPUExecutionProvider` is retained as a fallback. ONNX Runtime may also place unsupported operations on CPU.
- A listed CUDA provider does not guarantee that every model operation runs on the GPU or that GPU execution is faster.
- The repository's demo supports uploading a short reference recording and cloning it onto generated Persian speech.
- The model is **non-commercial (CC-BY-NC-4.0)**.